# Train the CNN guidance-map (off-machine, Colab/GPU)
Consumes a dataset built on the planner machine via `ml_planner.dataset_gen.export_dataset`.
**Hard I/O contract (must match `ml_planner/guidance.py`):** input `channels` (1,4,256,256) float32, output `cost_to_go` (1,1,256,256) float32, opset >= 11. Trains a small U-Net with a masked MSE loss (only labeled cells), then exports ONNX.
torch is used ONLY here; the planner never imports it.

In [ ]:
import numpy as np, torch, torch.nn as nn
GRID_RES = 256
data = np.load('dataset.npz')  # channels (N,4,H,W), label (N,H,W), mask (N,H,W)
channels = torch.tensor(data['channels']); label = torch.tensor(data['label']); mask = torch.tensor(data['mask'])

In [ ]:
class UNetSmall(nn.Module):
    def __init__(self, cin=4):
        super().__init__()
        self.enc = nn.Sequential(nn.Conv2d(cin,32,3,padding=1), nn.ReLU(), nn.Conv2d(32,32,3,padding=1), nn.ReLU())
        self.head = nn.Conv2d(32,1,1)
    def forward(self, x):
        return self.head(self.enc(x))
model = UNetSmall()

In [ ]:
opt = torch.optim.Adam(model.parameters(), 1e-3)
for epoch in range(50):
    opt.zero_grad()
    pred = model(channels)[:,0]
    m = mask > 0
    loss = (((pred - label)**2) * m).sum() / m.sum().clamp(min=1)  # masked MSE
    loss.backward(); opt.step()
print('final masked MSE', float(loss))

In [ ]:
dummy = torch.zeros(1,4,GRID_RES,GRID_RES)
torch.onnx.export(model, dummy, 'guidance.onnx', input_names=['channels'], output_names=['cost_to_go'], opset_version=13)
# Copy guidance.onnx to ml_planner/models/ on the planner machine.